In [ ]:
# Cluster task. Connects to Lakebase with a Databricks-minted OAuth token (no stored password).
import sys, os

ctx = dbutils.notebook.entry_point.getDbutils().notebook().getContext()
nb_dir = os.path.dirname(ctx.notebookPath().get())
if ("/Workspace" + nb_dir) not in sys.path:
    sys.path.append("/Workspace" + nb_dir)
from state_store import StateStore


def widget(name, default=""):
    dbutils.widgets.text(name, default)
    return dbutils.widgets.get(name)


db_instance = widget("db_instance", "lakefed-ingest-mt")
db_name = widget("db_name", "lakefed_ingest_mt")
store = StateStore.connect_lakebase(db_instance, db_name)

In [ ]:
# Read the per-table checkpoint and publish it for the downstream condition tasks + CT read.
src_database = widget("src_database")
src_schema = widget("src_schema")
src_table = widget("src_table")

cp = store.get_checkpoint(src_database, src_schema, src_table)
dbutils.jobs.taskValues.set(key="phase", value=cp["phase"])
dbutils.jobs.taskValues.set(key="ct_version", value=str(cp["ct_version"]))
print("checkpoint:", cp)